version 2. use definition to fine tune a model

In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# A full training

# Install

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [36]:
# !pip install datasets evaluate transformers[sentencepiece]
# !pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

# Check GPU is connected

In [37]:
import torch

In [38]:
# !nvidia-smi

In [39]:

if torch.cuda.is_available():
    print("CUDA is available! 🎉")
    print("Number of CUDA devices:", torch.cuda.device_count())
else:
    print("CUDA is not available. 🙁")

CUDA is not available. 🙁


In [40]:
from subprocess import call
import torch
import sys

print('__Python VERSION:', sys.version)

print('__pyTorch VERSION:', torch.__version__)

print('__CUDA VERSION')

# call(["nvcc", "--version"]) does not work

# ! nvcc --version

# print('__CUDNN VERSION:', torch.backends.cudnn.version())

# print('__Number CUDA Devices:', torch.cuda.device_count())

# print('__Devices')

# call(["nvidia-smi", "--format=csv", "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free"])

# print('Active CUDA Device: GPU', torch.cuda.current_device())

# print ('Available devices ', torch.cuda.device_count())

# print ('Current cuda device ', torch.cuda.current_device())

__Python VERSION: 3.11.8 (main, Jan 23 2025, 11:02:55) [Clang 15.0.0 (clang-1500.3.9.4)]
__pyTorch VERSION: 2.5.1
__CUDA VERSION


In [41]:
# activate GPU device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


# Load dataset

In [42]:
from pathlib import Path
import sys
import pandas as pd
# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

In [45]:
from datasets import load_dataset
from datasets import Dataset
from datasets import DatasetDict
import pandas as pd
import os

/Users/gl665/Documents/Phrame-MentalMaps-1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
parent_dir = Path(cwd).parent

# keep in mind that you need to use "/" instead of "\" in the path in windows machine
training_dataset_path = str(
    parent_dir / 'data/processed/SYNCH/training_set/training_set_0715_def_A0729.csv')

In [ ]:
# When training and test set is in one dataset
my_dataset = pd.read_excel(training_dataset_path)
my_dataset = my_dataset[['sentence1', 'sentence2', 'label', 'idx']]
# print(my_dataset.head())
print(my_dataset.shape)
# delete rows with missing values
my_dataset = my_dataset.dropna()
print(my_dataset.shape)
# print(my_dataset.isna().sum())

FileNotFoundError: [Errno 2] No such file or directory: '/Users/gl665/Documents/Phrame-MentalMaps-1/notebook/data/data_dictionary/training_set_forlabel_def_0715.xlsx'

In [13]:
# delete rows if the row does not equal to "1" or "0"
my_dataset = my_dataset[my_dataset['label'].isin(['0', '1'])]
print(my_dataset.shape)

NameError: name 'my_dataset' is not defined

In [386]:

# set datatype for label and idx as int
my_dataset['label'] = my_dataset['label'].astype(int)
my_dataset['idx'] = my_dataset['idx'].astype(int)

In [387]:
my_dataset.head(3)

,sentence1,sentence2,label,idx
0,using healing resources,providing resources to help police officers he...,0,0
1,using healing resources,experiencing unresolved trauma/pain that requi...,0,1
2,using healing resources,the extent to which indivduals trust resources...,0,2


In [388]:

my_dataset = Dataset.from_pandas(my_dataset, preserve_index=False)

train_test_dataset = my_dataset.train_test_split(test_size=0.5)
test_valid = train_test_dataset['test'].train_test_split(test_size=0.95)

# raw_train_dataset = raw_datasets["train"]

# gather everyone if you want to have a single DatasetDict
raw_datasets = DatasetDict({
    'train': train_test_dataset['train'],
    'test': test_valid['test'],
    'validation': test_valid['train']})

In [389]:

# # Load training set
# my_dataset = pd.read_csv(training_dataset_path)
# my_dataset = Dataset.from_pandas(my_dataset)

# # Load test set
# test_set = pd.read_csv(test_dataset_path)
# test_set = test_set.dropna() #TODO: droping NA for now, but there is a problem with the test set
# test_dataset = Dataset.from_pandas(test_set)

# train_test_dataset = my_dataset.train_test_split(test_size=0.1, shuffle=True)
# # test_valid = train_test_dataset['test'].train_test_split(test_size=0.8)

# # raw_train_dataset = raw_datasets["train"]

# # gather everyone if you want to have a single DatasetDict
# raw_datasets = DatasetDict({
#     'train': train_test_dataset['train'],
#     'validation': train_test_dataset['test'],
#     'test': test_dataset})

In [390]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1891
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1797
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 94
    })
})

In [391]:
dataset_folder_path = 'models/fine_tune_model/03_training_set_0711_var'

cwd = os.getcwd()

parent_dir = Path(cwd).parent


folder_path = str(parent_dir / dataset_folder_path)
print(folder_path)

c:\Users\GL665\Code\Phrame-MentalMaps\models\fine_tune_model\03_training_set_0711_var


In [392]:
dataset_size = {k: len(v) for k, v in raw_datasets.items()}

# Fine tuning a model

In [393]:
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModel
import torch
import torch.nn.functional as F

In [394]:
checkpoint = 'sentence-transformers/all-mpnet-base-v2'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

model.to(device)  # Move model to the device

c:\Users\GL665\Code\Phrame-MentalMaps\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [395]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 94/94 [00:00<00:00, 5474.83 examples/s]


# prepare for training

In [396]:
tokenized_datasets = tokenized_datasets.remove_columns(
    ["sentence1", "sentence2", "idx"])

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

tokenized_datasets.set_format("torch")

tokenized_datasets['train'].column_names

['labels', 'input_ids', 'attention_mask']

In [397]:
# samples = tokenized_datasets["train"][:8]
# samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
# [len(x) for x in samples["input_ids"]]

In [398]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=8, collate_fn=data_collator
)

In [399]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 66]),
 'attention_mask': torch.Size([8, 66])}

In [400]:
# Move the batch of data to the same device
for key in batch.keys():
    batch[key] = batch[key].to(device)

# train the model

## train with trainer

In [401]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=os.path.join(parent_dir, 'models/fine_tune_model'))

In [402]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2)

Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [403]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [404]:
trainer.train()

 70%|███████   | 500/711 [02:33<01:06,  3.18it/s]

{'loss': 0.3571, 'grad_norm': 1.5571473836898804, 'learning_rate': 1.4838255977496484e-05, 'epoch': 2.11}


100%|██████████| 711/711 [03:42<00:00,  3.20it/s]

{'train_runtime': 222.2746, 'train_samples_per_second': 25.522, 'train_steps_per_second': 3.199, 'train_loss': 0.32952787503914494, 'epoch': 3.0}


TrainOutput(global_step=711, training_loss=0.32952787503914494, metrics={'train_runtime': 222.2746, 'train_samples_per_second': 25.522, 'train_steps_per_second': 3.199, 'total_flos': 212099371724520.0, 'train_loss': 0.32952787503914494, 'epoch': 3.0})

Transform the logits to real predictions

In [405]:
predictions = trainer.predict(tokenized_datasets["test"])
print(predictions.predictions.shape, predictions.label_ids.shape)

100%|██████████| 225/225 [00:20<00:00, 11.07it/s]

(1797, 2) (1797,)


using softmax to get the probability of the class. first column is when the class is 0, second column is when the class is 1.. 

In [406]:
from scipy.special import softmax
probs = softmax(predictions.predictions, axis=1)
print(probs[1:20])

[[0.96350557 0.0364944 ]
 [0.90828973 0.09171031]
 [0.8845908  0.11540925]
 [0.96682626 0.03317379]
 [0.9663745  0.03362545]
 [0.95357233 0.04642769]
 [0.3930087  0.60699135]
 [0.9249482  0.07505178]
 [0.9675309  0.03246912]
 [0.9646507  0.03534937]
 [0.9507232  0.04927688]
 [0.9517838  0.04821619]
 [0.96721005 0.03278997]
 [0.07267775 0.92732227]
 [0.8133832  0.18661681]
 [0.96533835 0.03466165]
 [0.9671407  0.03285934]
 [0.91204107 0.08795892]
 [0.9594702  0.04052979]]


use argmax to get the final predicted class (where the class would be either 0 or 1)

In [407]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)
print(preds[1:20])

[0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0]


## Print validation set

In [408]:
validation_dataset = tokenized_datasets["validation"]

In [409]:
raw_validation_dataset = raw_datasets["validation"]
raw_validation_dataset[0]

{'sentence1': 'Barriers that increase difficulty for individuals to fufill DSS requirements for reunification - typically discussed in terms of ability to access resources, navigate social service systems, obtain finances/housing to meet requirements, and juggle multiple obligations',
 'sentence2': 'legal suspension of parental custody - children being placed in the custody of others (e.g., other family members, foster homes, group homes)',
 'label': 0,
 'idx': 1823}

# evaluate

In [410]:
import evaluate
metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])

# metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

results = metric.compute()
print(results)

{'accuracy': 0.887590428491931, 'f1': 0.5429864253393665, 'precision': 0.7547169811320755, 'recall': 0.42402826855123676}


# Save results

In [411]:
# import experiment_logger to juptter notebook
import sys
sys.path.append(str(parent_dir / 'src/model_script'))

print(sys.path)

['C:\\Users\\GL665\\AppData\\Local\\Programs\\Python\\Python311\\python311.zip', 'C:\\Users\\GL665\\AppData\\Local\\Programs\\Python\\Python311\\DLLs', 'C:\\Users\\GL665\\AppData\\Local\\Programs\\Python\\Python311\\Lib', 'C:\\Users\\GL665\\AppData\\Local\\Programs\\Python\\Python311', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\.venv', '', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\.venv\\Lib\\site-packages', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\.venv\\Lib\\site-packages\\win32', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\.venv\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\.venv\\Lib\\site-packages\\Pythonwin', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps', 'C:\\Users\\GL665\\.cache\\huggingface\\modules', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\src', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\src\\model_script', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\src\\model_script', 'c:\\Users\\GL665\\Code\\Phrame-MentalMaps\\src\\model_script', 'c

In [412]:
from experiment_logger import save_run_results

result_path = os.path.join(parent_dir, 'models/fine_tune_model')

save_run_results(training_dataset_path, checkpoint, dataset_size, results, result_path,
                 trainer, model)

# Inference

Let's test the model on a new sentence:

In [413]:
raw_datasets['validation'][0]

{'sentence1': 'Barriers that increase difficulty for individuals to fufill DSS requirements for reunification - typically discussed in terms of ability to access resources, navigate social service systems, obtain finances/housing to meet requirements, and juggle multiple obligations',
 'sentence2': 'legal suspension of parental custody - children being placed in the custody of others (e.g., other family members, foster homes, group homes)',
 'label': 0,
 'idx': 1823}

In [414]:
dataset_folder_path = 'models/fine_tune_model/03_training_set_0711_var'

cwd = os.getcwd()

parent_dir = Path(cwd).parent


folder_path = str(parent_dir / dataset_folder_path)
print(folder_path)

c:\Users\GL665\Code\Phrame-MentalMaps\models\fine_tune_model\03_training_set_0711_var


In [415]:
sentence1 = "child welfare system intervention"
sentence2 = "children removed from house"
sentence3 = "I am climbing and looking for a climbing partner"

In [416]:

encoding = tokenizer(sentence1, sentence2, return_tensors="pt")
encoding = {k: v.to(trainer.model.device) for k, v in encoding.items()}

outputs = trainer.model(**encoding)
print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[ 1.5998, -1.6366]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


The logits that come out of the model are of shape (batch_size, num_labels). As we are only forwarding a single sentence through the model, the `batch_size` equals 1. The logits is a tensor that contains the (unnormalized) scores for every individual label.

To turn them into actual predicted labels, we first apply a sigmoid function independently to every score, such that every score is turned into a number between 0 and 1, that can be interpreted as a "probability" for how certain the model is that a given class belongs to the input text.

Next, we use a threshold (typically, 0.5) to turn every probability into either a 1 (which means, we predict the label for the given example) or a 0 (which means, we don't predict the label for the given example).